In [2]:
from huggingface_hub import login
login()

In [3]:
# @title 1. Install Dependencies & Build FastAlign
import os

print("⏳ Installing Python Libraries...")
!pip install -q transformers sentence-transformers datasets torch seaborn matplotlib scikit-learn

print("⏳ Building FastAlign (Statistical Model)...")
# Clone and compile FastAlign C++ source for the statistical part of our ensemble
!git clone https://github.com/clab/fast_align.git > /dev/null 2>&1
!mkdir -p fast_align/build
%cd fast_align/build
!cmake .. > /dev/null 2>&1
!make > /dev/null 2>&1
%cd ../..

print("✅ Installation Complete.")

⏳ Installing Python Libraries...
⏳ Building FastAlign (Statistical Model)...
/content/fast_align/build
/content
✅ Installation Complete.


In [4]:
# @title 2. Import Libraries & Check GPU
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from datasets import load_dataset
from IPython.display import display, HTML
import itertools

# Optimize for T4 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Hardware Accelerator: {device}")

✅ Hardware Accelerator: cuda


In [5]:
# @title 3. Load English-Malayalam Dataset
print("⏳ Loading FLORES+ Dataset...")

try:
    # Load separate language subsets
    ds_eng = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="dev", trust_remote_code=True)
    ds_mal = load_dataset("openlanguagedata/flores_plus", "mal_Mlym", split="dev", trust_remote_code=True)

    # Create pairs (FLORES is sentence-aligned by index)
    pairs = []
    # Taking first 10 sentences for rapid testing
    for i in range(10):
        pairs.append((ds_eng['text'][i], ds_mal['text'][i]))

    print(f"✅ Loaded {len(pairs)} pairs successfully.")
    print(f"📝 Example: {pairs[0][0]} <--> {pairs[0][1]}")

except Exception as e:
    print(f"⚠️ Error loading dataset: {e}. Using fallback sample data.")
    pairs = [
        ("I am going to the library.", "ഞാൻ ലൈബ്രറിയിലേക്ക് പോകുന്നു."),
        ("The quick brown fox jumps over the lazy dog.", "വേഗതയുള്ള തവിട്ടുനിറമുള്ള കുറുക്കൻ മടിയനായ നായയുടെ മുകളിലൂടെ ചാടുന്നു.")
    ]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'openlanguagedata/flores_plus' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'openlanguagedata/flores_plus' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


⏳ Loading FLORES+ Dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/74.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

...

dev/mal_Mlym.parquet:   0%|          | 0.00/183k [00:00<?, ?B/s]

devtest/mal_Mlym.parquet:   0%|          | 0.00/192k [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/997 [00:00<?, ? examples/s]

Generating devtest split:   0%|          | 0/1012 [00:00<?, ? examples/s]

✅ Loaded 10 pairs successfully.
📝 Example: On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each. <--> തിങ്കളാഴ്ച്ച, സ്റ്റാൻഫോർഡ് യൂണിവേഴ്‌സിറ്റി സ്‌കൂൾ ഓഫ് മെഡിസിനിലെ ശാസ്ത്രജ്ഞന്മാർ കോശങ്ങളെ അവയുടെ ഇനം അനുസരിച്ച് തരംതിരിക്കാൻ കഴിയുന്ന ഒരു പുതിയ രോഗനിർണയ ഉപകരണം കണ്ടുപിടിച്ചതായി പ്രഖ്യാപിച്ചു: സ്റ്റാൻഡേർഡ് ഇങ്ക്‌ജെറ്റ് പ്രിന്റ്ററുകൾ ഉപയോഗിച്ച് നിർമ്മിക്കാൻ സാധിക്കുന്ന ഏകദേശം ഒരു യു.എസ് സെന്റ് ഓരോന്നിനും വേണ്ടിവരുന്ന പ്രിന്റ് ചെയ്യാൻ കഴിയുന്ന ഒരു ചെറിയ ചിപ്പ്.


In [6]:
# @title 4. Load Neural Models
print("⏳ Loading LaBSE (SimAlign)...")
sim_model = SentenceTransformer('sentence-transformers/LaBSE').to(device)

print("⏳ Loading mBERT (Awesome-Align)...")
awesome_id = "bert-base-multilingual-cased"
awesome_tokenizer = AutoTokenizer.from_pretrained(awesome_id)
awesome_model = AutoModel.from_pretrained(awesome_id).to(device)
awesome_model.eval()

print("✅ Models Loaded to GPU.")

⏳ Loading LaBSE (SimAlign)...


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

✅ Models Loaded to GPU.


In [7]:
# @title 5. Define Alignment Algorithms (SimAlign, Awesome, FastAlign)

# --- Method A: SimAlign (LaBSE) ---
def run_simalign(eng, mal):
    e_words = eng.split()
    m_words = mal.split()

    e_emb = sim_model.encode(e_words, convert_to_tensor=True).cpu().numpy()
    m_emb = sim_model.encode(m_words, convert_to_tensor=True).cpu().numpy()

    sim_matrix = cosine_similarity(e_emb, m_emb)

    aligns = []
    for i, row in enumerate(sim_matrix):
        j = np.argmax(row)
        score = row[j]
        aligns.append((i, j, score))

    return e_words, m_words, aligns

# --- Method B: Awesome-Align (mBERT - FIXED) ---
def run_awesome(eng, mal):
    # 1. Split manually first so we control word boundaries
    e_words = eng.split()
    m_words = mal.split()

    # 2. Tokenize the LIST (is_split_into_words=True)
    # This prevents the "punctuation as separate word" bug that caused 114% coverage
    inp_src = awesome_tokenizer(e_words, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=512).to(device)
    inp_tgt = awesome_tokenizer(m_words, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        out_src = awesome_model(**inp_src, output_hidden_states=True)
        out_tgt = awesome_model(**inp_tgt, output_hidden_states=True)
        src_emb = out_src.hidden_states[8][0]
        tgt_emb = out_tgt.hidden_states[8][0]

        sim_mat = torch.matmul(src_emb, tgt_emb.t())
        probs = torch.nn.functional.softmax(sim_mat, dim=-1).cpu().numpy()

    w_src = inp_src.word_ids()
    w_tgt = inp_tgt.word_ids()
    threshold = 1e-3

    aligns = []
    indices = np.argwhere(probs > threshold)
    seen = {}

    for i, j in indices:
        if w_src[i] is None or w_tgt[j] is None: continue
        wi, wj = w_src[i], w_tgt[j]
        score = probs[i, j]
        # Keep MAX score for each word pair
        if (wi, wj) not in seen or score > seen[(wi, wj)]:
            seen[(wi, wj)] = score

    # Return list of results
    return e_words, m_words, [(k[0], k[1], v) for k, v in seen.items()]

# --- Method C: FastAlign (Statistical) ---
def run_fastalign(pair):
    eng, mal = pair
    with open("input.txt", "w") as f: f.write(f"{eng} ||| {mal}\n")

    !./fast_align/build/fast_align -i input.txt -d -o -v > fwd.align 2>/dev/null
    !./fast_align/build/fast_align -i input.txt -d -o -v -r > rev.align 2>/dev/null
    !./fast_align/build/atools -i fwd.align -j rev.align -c grow-diag-final-and > final.align 2>/dev/null

    res = []
    if os.path.exists("final.align"):
        with open("final.align") as f:
            for line in f:
                for item in line.split():
                    i, j = map(int, item.split('-'))
                    res.append((i, j, 1.0))
    return res

In [8]:
# @title 6. Run Ensemble Pipeline
def run_ensemble(idx):
    if idx >= len(pairs): return None
    eng, mal = pairs[idx]

    # 1. Run all three
    _, _, s_res = run_simalign(eng, mal)
    _, _, a_res = run_awesome(eng, mal)
    f_res = run_fastalign((eng, mal))

    # 2. Vote
    votes = {}
    for i, j, _ in a_res: votes[(i,j)] = votes.get((i,j), 0) + 1.2
    for i, j, _ in s_res: votes[(i,j)] = votes.get((i,j), 0) + 0.8
    for i, j, _ in f_res: votes[(i,j)] = votes.get((i,j), 0) + 0.5

    # 3. Consensus Filter
    ens_res = []
    for (i,j), score in votes.items():
        if score >= 1.0:
            # Normalize score to be somewhat readable (0-100% scale approximate)
            normalized = min(1.0, score / 2.5)
            ens_res.append((i, j, normalized))

    return eng.split(), mal.split(), s_res, a_res, ens_res

In [9]:
# @title 7. Define SVG Visualization
def render_svg(src, tgt, aligns, title, color_hex):
    width, height = 800, 220
    src_gap = (width-100)/max(1, len(src)-1)
    tgt_gap = (width-100)/max(1, len(tgt)-1)

    svg = [f'<div style="overflow-x:auto"><svg width="{width}" height="{height}" style="background:white; border:1px solid #eee; border-radius:8px; margin-bottom:20px">']
    svg.append(f'<text x="50%" y="30" text-anchor="middle" font-family="Arial" font-weight="bold" fill="#333">{title}</text>')

    src_x = []
    for i, w in enumerate(src):
        x = 50 + i*src_gap
        src_x.append(x)
        svg.append(f'<text x="{x}" y="70" text-anchor="middle" font-size="14" fill="#000">{w}</text>')
        svg.append(f'<circle cx="{x}" cy="85" r="4" fill="#555"/>')

    tgt_x = []
    for i, w in enumerate(tgt):
        x = 50 + i*tgt_gap
        tgt_x.append(x)
        svg.append(f'<text x="{x}" y="180" text-anchor="middle" font-size="14" fill="#000">{w}</text>')
        svg.append(f'<circle cx="{x}" cy="165" r="4" fill="#555"/>')

    for i, j, score in aligns:
        if i < len(src_x) and j < len(tgt_x):
            op = min(1.0, max(0.2, score))
            path = f'<path d="M{src_x[i]},85 C{src_x[i]},125 {tgt_x[j]},125 {tgt_x[j]},165" stroke="{color_hex}" stroke-width="2" fill="none" opacity="{op}"/>'
            svg.append(path)

    svg.append('</svg></div>')
    return HTML("".join(svg))

In [10]:
# @title 8. Run Final Dashboard
res = run_ensemble(0) # Change to 1, 2, 3 to see other sentences

if res:
    e_words, m_words, s_res, a_res, ens_res = res

    print(f"📊 ANALYSIS FOR PAIR #0")
    print(f"🇬🇧: {' '.join(e_words)}")
    print(f"🇮🇳: {' '.join(m_words)}")
    print("-" * 60)

    display(render_svg(e_words, m_words, s_res, "❌ Baseline: SimAlign", "#ff4d4d"))
    display(render_svg(e_words, m_words, a_res, "✅ Improved: Awesome-Align", "#0066cc"))
    display(render_svg(e_words, m_words, ens_res, "🏆 Best Outcome: Ensemble", "#00cc66"))

    print("\n📈 FINAL STATISTICS")
    data = {
        "Metric": ["Avg Confidence", "Alignments Found", "Coverage"],
        "SimAlign": [
            f"{np.mean([x[2] for x in s_res]):.2f}" if s_res else "0.0",
            len(s_res),
            f"{len(set(x[0] for x in s_res))/len(e_words):.1%}"
        ],
        "Awesome-Align": [
            f"{np.mean([x[2] for x in a_res]):.2f}" if a_res else "0.0",
            len(a_res),
            f"{len(set(x[0] for x in a_res))/len(e_words):.1%}"
        ],
        "Ensemble (Best)": [
            f"{np.mean([x[2] for x in ens_res]):.2f}" if ens_res else "0.0",
            len(ens_res),
            f"{len(set(x[0] for x in ens_res))/len(e_words):.1%}"
        ]
    }
    display(pd.DataFrame(data))

📊 ANALYSIS FOR PAIR #0
🇬🇧: On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.
🇮🇳: തിങ്കളാഴ്ച്ച, സ്റ്റാൻഫോർഡ് യൂണിവേഴ്‌സിറ്റി സ്‌കൂൾ ഓഫ് മെഡിസിനിലെ ശാസ്ത്രജ്ഞന്മാർ കോശങ്ങളെ അവയുടെ ഇനം അനുസരിച്ച് തരംതിരിക്കാൻ കഴിയുന്ന ഒരു പുതിയ രോഗനിർണയ ഉപകരണം കണ്ടുപിടിച്ചതായി പ്രഖ്യാപിച്ചു: സ്റ്റാൻഡേർഡ് ഇങ്ക്‌ജെറ്റ് പ്രിന്റ്ററുകൾ ഉപയോഗിച്ച് നിർമ്മിക്കാൻ സാധിക്കുന്ന ഏകദേശം ഒരു യു.എസ് സെന്റ് ഓരോന്നിനും വേണ്ടിവരുന്ന പ്രിന്റ് ചെയ്യാൻ കഴിയുന്ന ഒരു ചെറിയ ചിപ്പ്.
------------------------------------------------------------



📈 FINAL STATISTICS


,Metric,SimAlign,Awesome-Align,Ensemble (Best)
0,Avg Confidence,0.80,0.76,0.62
1,Alignments Found,43,63,64
2,Coverage,100.0%,100.0%,100.0%


In [11]:
# @title 📊 Calculate "Pseudo-Accuracy" (Precision Score)
def calculate_accuracy_metrics(idx):
    if idx >= len(pairs): return

    # 1. Run Pipeline to get all results
    _, _, s_res, a_res, ens_res = run_ensemble(idx)

    # 2. Convert to Sets for comparison
    # We only care about the connections (Source_Index, Target_Index)
    s_set = set((x[0], x[1]) for x in s_res)
    a_set = set((x[0], x[1]) for x in a_res)
    ens_set = set((x[0], x[1]) for x in ens_res) # This is our "Truth"

    # 3. Calculate Precision (Accuracy relative to Ensemble)
    # Formula: (Matches that appear in Ensemble) / (Total Matches found by Model)

    # SimAlign Accuracy
    s_correct = len(s_set.intersection(ens_set))
    s_total = len(s_set) if len(s_set) > 0 else 1
    s_acc = (s_correct / s_total) * 100

    # Awesome-Align Accuracy
    a_correct = len(a_set.intersection(ens_set))
    a_total = len(a_set) if len(a_set) > 0 else 1
    a_acc = (a_correct / a_total) * 100

    print(f"🏆 SCORING REPORT (Sentence #{idx})")
    print("-" * 40)
    print(f"✅ Ensemble (Truth) found: {len(ens_set)} valid connections")
    print("-" * 40)

    print(f"1️⃣  SimAlign Performance:")
    print(f"    - Guesses Made: {s_total}")
    print(f"    - Correct Guesses: {s_correct}")
    print(f"    - ERROR RATE: {100 - s_acc:.1f}% (Noise)")
    print(f"    - PRECISION SCORE: {s_acc:.1f}%")
    print("-" * 40)

    print(f"2️⃣  Awesome-Align Performance:")
    print(f"    - Guesses Made: {a_total}")
    print(f"    - Correct Guesses: {a_correct}")
    print(f"    - ERROR RATE: {100 - a_acc:.1f}% (Noise)")
    print(f"    - PRECISION SCORE: {a_acc:.1f}%")
    print("-" * 40)

    if a_acc > s_acc:
        print("🎉 CONCLUSION: Awesome-Align is more ACCURATE.")
    else:
        print("⚖️ CONCLUSION: SimAlign is more ACCURATE (Rare).")

# Run the Score Calculator
calculate_accuracy_metrics(0)

🏆 SCORING REPORT (Sentence #0)
----------------------------------------
✅ Ensemble (Truth) found: 64 valid connections
----------------------------------------
1️⃣  SimAlign Performance:
    - Guesses Made: 43
    - Correct Guesses: 27
    - ERROR RATE: 37.2% (Noise)
    - PRECISION SCORE: 62.8%
----------------------------------------
2️⃣  Awesome-Align Performance:
    - Guesses Made: 63
    - Correct Guesses: 63
    - ERROR RATE: 0.0% (Noise)
    - PRECISION SCORE: 100.0%
----------------------------------------
🎉 CONCLUSION: Awesome-Align is more ACCURATE.
